In [1]:
from pathlib import Path

import numpy
from astropy.io import fits
from matplotlib import pyplot
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pandas import read_csv

ALL_DATASETS_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid")
RESULTS_SEXTRACTOR_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR")

In [2]:
VERSION = "DATA_1"

DATASET_DIR = ALL_DATASETS_DIR / VERSION
IMAGES_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "IMAGES"
VIGNET_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "VIGNET"
RESULTS_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "RESULTS"

# psf

In [ ]:
img_psf = fits.open(ALL_DATASETS_DIR / "PSF" / "airy.psf.fits")[0].data
fig, axes = pyplot.subplots(1, 2, figsize=(12, 6))
im1 = axes[0].imshow(numpy.log(img_psf), cmap="gray")
axes[0].set_title("Airy Logarithmic Scale")
divider = make_axes_locatable(axes[0])
cax1 = divider.append_axes("right", size="5%", pad=0.05)
pyplot.colorbar(im1, cax=cax1)
im2 = axes[1].imshow(img_psf, cmap="gray")
axes[1].set_title("Airy Linear Scale")
divider = make_axes_locatable(axes[1])
cax2 = divider.append_axes("right", size="5%", pad=0.05)
pyplot.colorbar(im2, cax=cax2)
pyplot.show()

In [4]:
def print_fits(name, title, divide=1):
    img = fits.open(f"{name}.fits")[0].data
    pyplot.figure()
    pyplot.title(f"{title} => Min: {numpy.min(img):.3f}, Max: {numpy.max(img):.3f}")
    pyplot.imshow(img, cmap="gray", vmax=numpy.max(img) / divide)
    pyplot.colorbar()
    pyplot.show()

# images

In [ ]:
CHOICE = "simu_2000"

print_fits(name=DATASET_DIR / CHOICE, title="image")
print_fits(IMAGES_DIR / f"segmentation_{CHOICE}", title="segmentation")
print_fits(IMAGES_DIR / f"background_{CHOICE}", title="background")
print_fits(IMAGES_DIR / f"residual_{CHOICE}_1", title="residual")
print_fits(IMAGES_DIR / f"fitting_{CHOICE}_1", title="fitting")

# results

In [6]:
RESULTS = sorted(list((RESULTS_DIR).glob("simu_*.csv")))
avg_e1, avg_e2 = [], []
for sample in RESULTS:
    e1 = read_csv(sample)["e1"]
    e2 = read_csv(sample)["e2"]
    avg_e1.append(e1.mean())
    avg_e2.append(e2.mean())
numpy.savetxt(
    RESULTS_DIR / "avg_ellipticies.csv",
    numpy.column_stack((avg_e1, avg_e2)),
    delimiter=",",
    header="avg_e1,avg_e2",
)

# vignettes

In [7]:
def plot_vignettes(vignettes_images, pos_x, pos_y):
    _, axes = pyplot.subplots(3, 4, figsize=(12, 9))
    axes = axes.flatten()

    for i, (fits_file, x, y) in enumerate(zip(vignettes_images[:12], pos_x, pos_y)):
        image = fits.open(fits_file)[0].data
        img = axes[i].imshow(image, cmap="gray")
        axes[i].set_title(f"{fits_file.stem} {x:.2f} {y:.2f}")
        divider = make_axes_locatable(axes[i])
        cax1 = divider.append_axes("right", size="5%", pad=0.05)
        pyplot.colorbar(img, cax=cax1)
        axes[i].axis("off")

    for j in range(len(vignettes_images), 12):
        axes[j].axis("off")

    pyplot.tight_layout()
    pyplot.show()

In [8]:
CHOICE = "simu_1000"
X = read_csv(f"{RESULTS_DIR / CHOICE}.csv")["x"]
Y = read_csv(f"{RESULTS_DIR / CHOICE}.csv")["y"]
VIGNETTES = sorted(list((VIGNET_DIR / CHOICE).glob("*.fits")))

In [ ]:
plot_vignettes(VIGNETTES, X, Y)